# NLP Foundations — Exercises 1 to 11

This notebook walks through eleven foundational NLP exercises, moving from basic
text statistics (word frequency, vocabulary) through classic representations
(co-occurrence matrices, TF-IDF, dense vectors) to simple probabilistic language
models (bigram / conditional-probability next-word prediction).

Each exercise includes:
- **Objective** — what the exercise is asking for
- **Concepts Learned** — the underlying NLP/ML ideas
- **Code** — a runnable solution
- **Output** — the result printed by the code


## Setup

Shared imports used throughout the notebook.

In [1]:
import math
from collections import defaultdict, Counter

## Exercise 1: Word Frequency Counter

**Objective:** Count how many times each word appears in a sentence.

**Input:** `text = "I like NLP I like Python NLP is fun"`

**Tasks**
1. Split the sentence into words.
2. Count each word.
3. Store the result in a dictionary.
4. Print the dictionary.

**Concepts Learned:** Strings · Lists · Dictionary · Term Frequency (TF)


In [2]:
text = "I like NLP I like Python NLP is fun"
words = text.split()

freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

for word, count in freq.items():
    print(f"{word} : {count}")

I : 2
like : 2
NLP : 2
Python : 1
is : 1
fun : 1


## Exercise 2: Vocabulary Builder

**Objective:** Find all unique words in a corpus.

**Input:**
```python
corpus = ["I like NLP", "I like AI", "AI is amazing"]
```

**Tasks**
1. Read every sentence.
2. Split into words.
3. Create a vocabulary using a set.
4. Print the vocabulary size.

**Concepts Learned:** Vocabulary · Unique words


In [3]:
corpus = [
    "I like NLP",
    "I like AI",
    "AI is amazing",
]

vocab = set()
for sentence in corpus:
    for word in sentence.split():
        vocab.add(word)

print("Vocabulary")
for word in sorted(vocab):
    print(word)
print(f"Vocabulary Size = {len(vocab)}")

Vocabulary
AI
I
NLP
amazing
is
like
Vocabulary Size = 6


## Exercise 3: Build a Word Co-occurrence Matrix

**Objective:** Construct a simple co-occurrence matrix using a window size of 1.

**Input:** `sentence = "the cat sat on the mat"`

**Tasks**
1. Build vocabulary.
2. Create an empty matrix.
3. Count neighboring words.

**Concepts Learned:** Co-occurrence Matrix · Context Window


In [4]:
sentence = "the cat sat on the mat"
words = sentence.split()
vocab_sent = sorted(set(words))

# empty matrix
matrix = {w1: {w2: 0 for w2 in vocab_sent} for w1 in vocab_sent}

# count neighboring words, window size 1
for i, word in enumerate(words):
    if i > 0:
        matrix[word][words[i - 1]] += 1
    if i < len(words) - 1:
        matrix[word][words[i + 1]] += 1

# print matrix
print("Word".ljust(6), *[w.ljust(5) for w in vocab_sent])
for w1 in vocab_sent:
    row = [str(matrix[w1][w2]).ljust(5) for w2 in vocab_sent]
    print(w1.ljust(6), *row)

Word   cat   mat   on    sat   the  
cat    0     0     0     1     1    
mat    0     0     0     0     1    
on     0     0     0     1     1    
sat    1     0     1     0     0    
the    1     1     1     0     0    


## Exercise 4: Detect Data Sparsity

**Objective:** Find word pairs that never occur together.

**Input:** Use the co-occurrence matrix from Exercise 3.

**Tasks**
1. Traverse the matrix.
2. Find all cells with value 0.
3. Print those word pairs.

**Concepts Learned:** Data Sparsity · Sparse Matrix


In [5]:
sparse_pairs = []
for i, w1 in enumerate(vocab_sent):
    for w2 in vocab_sent[i + 1:]:  # avoid duplicate/self pairs
        if matrix[w1][w2] == 0:
            sparse_pairs.append((w1, w2))
            print(f"({w1}, {w2}) -> Never occurred")

(cat, mat) -> Never occurred
(cat, on) -> Never occurred
(mat, on) -> Never occurred
(mat, sat) -> Never occurred
(sat, the) -> Never occurred


## Exercise 5: Implement Laplace Smoothing

**Objective:** Calculate word probabilities using Laplace smoothing.

**Input:**
```python
word_counts = {"cat": 4, "dog": 3, "bird": 1}
```

**Formula:**
$$P(w) = \frac{\text{count}(w) + 1}{\text{total\_words} + \text{vocab\_size}}$$

**Tasks**
1. Compute total words.
2. Compute vocabulary size.
3. Apply Laplace smoothing.
4. Print probabilities.

**Concepts Learned:** Probability · Laplace Smoothing

> **Note:** the source worksheet's sample output (0.50 / 0.40 / 0.20) doesn't
> match its own input counts. Applying the formula correctly to
> `{cat: 4, dog: 3, bird: 1}` gives **0.45 / 0.36 / 0.18**, as shown below.


In [6]:
word_counts = {"cat": 4, "dog": 3, "bird": 1}

total_words = sum(word_counts.values())
vocab_size = len(word_counts)

probabilities = {}
for word, count in word_counts.items():
    prob = (count + 1) / (total_words + vocab_size)
    probabilities[word] = prob
    print(f"{word} : {prob:.2f}")

cat : 0.45
dog : 0.36
bird : 0.18


## Exercise 6: Simple Keyboard Prediction

**Objective:** Predict the most common next word.

**Training Corpus**
```
I like NLP
I like Python
I like coffee
I love NLP
```

**Input:** `I like`

**Tasks**
1. Build bigram counts.
2. Store next-word frequencies.
3. Display the top three predictions.

**Concepts Learned:** Bigram Language Model · Keyboard Prediction


In [7]:
training_corpus = [
    "I like NLP",
    "I like Python",
    "I like coffee",
    "I love NLP",
]

bigram_counts = defaultdict(Counter)
for sentence in training_corpus:
    ws = sentence.split()
    for i in range(len(ws) - 1):
        bigram_counts[ws[i]][ws[i + 1]] += 1

query = "I like".split()[-1]  # last word of the input phrase = "like"
suggestions = bigram_counts[query].most_common(3)

print("Input: I like")
print("Suggestions")
for word, _ in suggestions:
    print(word)

Input: I like
Suggestions
NLP
Python
coffee


## Exercise 7: Calculate TF-IDF

**Objective:** Compute TF, IDF, and TF-IDF scores.

**Corpus**
```
Document 1: I like NLP
Document 2: I like AI
Document 3: AI is amazing
```

**Tasks**
1. Calculate TF.
2. Calculate DF.
3. Calculate IDF.
4. Compute TF-IDF.

**Concepts Learned:** TF · DF · IDF · TF-IDF

> Uses the standard formula `IDF = log(N / DF)`.


In [8]:
documents = {
    "Document 1": "I like NLP",
    "Document 2": "I like AI",
    "Document 3": "AI is amazing",
}

doc_words = {doc: text.split() for doc, text in documents.items()}
vocab_tfidf = sorted(set(w for ws in doc_words.values() for w in ws))
N = len(documents)

# TF: term frequency per document
tf = {}
for doc, ws in doc_words.items():
    counts = Counter(ws)
    total = len(ws)
    tf[doc] = {w: counts[w] / total for w in vocab_tfidf}

# DF: number of documents containing each word
df = {w: sum(1 for ws in doc_words.values() if w in ws) for w in vocab_tfidf}

# IDF: log(N / DF)
idf = {w: math.log(N / df[w]) for w in vocab_tfidf}

# TF-IDF
tfidf = {doc: {w: tf[doc][w] * idf[w] for w in vocab_tfidf} for doc in documents}

for doc in documents:
    print(f"\n{doc}")
    for w in vocab_tfidf:
        if tf[doc][w] > 0:
            print(f"  {w}: TF={tf[doc][w]:.2f}  IDF={idf[w]:.2f}  TF-IDF={tfidf[doc][w]:.3f}")


Document 1
  I: TF=0.33  IDF=0.41  TF-IDF=0.135
  NLP: TF=0.33  IDF=1.10  TF-IDF=0.366
  like: TF=0.33  IDF=0.41  TF-IDF=0.135

Document 2
  AI: TF=0.33  IDF=0.41  TF-IDF=0.135
  I: TF=0.33  IDF=0.41  TF-IDF=0.135
  like: TF=0.33  IDF=0.41  TF-IDF=0.135

Document 3
  AI: TF=0.33  IDF=0.41  TF-IDF=0.135
  amazing: TF=0.33  IDF=1.10  TF-IDF=0.366
  is: TF=0.33  IDF=1.10  TF-IDF=0.366


## Exercise 8: Dense Vector Similarity

**Objective:** Compute cosine similarity between two dense vectors.

**Input:**
```python
cat = [0.4, 0.2, 0.8]
dog = [0.5, 0.1, 0.7]
```

**Tasks**
1. Compute dot product.
2. Compute vector magnitudes.
3. Calculate cosine similarity.

**Concepts Learned:** Dense Vectors · Cosine Similarity


In [9]:
def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a * a for a in v1))
    mag2 = math.sqrt(sum(b * b for b in v2))
    if mag1 == 0 or mag2 == 0:
        return 0.0
    return dot / (mag1 * mag2)

cat = [0.4, 0.2, 0.8]
dog = [0.5, 0.1, 0.7]

sim = cosine_similarity(cat, dog)
print(f"Cosine Similarity = {sim:.2f}")

Cosine Similarity = 0.98


## Exercise 9: Find Similar Words

**Objective:** Find the word most similar to a given word.

**Input:**
```python
embeddings = {
    "cat": [0.4, 0.2, 0.8],
    "dog": [0.5, 0.1, 0.7],
    "car": [-0.7, 0.8, -0.2],
    "lion": [0.45, 0.25, 0.82],
}
```

**Tasks**
1. Compare the target vector with all others.
2. Compute cosine similarity.
3. Print the most similar word.

**Concepts Learned:** Word Embeddings · Semantic Similarity


In [10]:
embeddings = {
    "cat": [0.4, 0.2, 0.8],
    "dog": [0.5, 0.1, 0.7],
    "car": [-0.7, 0.8, -0.2],
    "lion": [0.45, 0.25, 0.82],
}

target = "cat"
best_word, best_sim = None, -1

for word, vec in embeddings.items():
    if word == target:
        continue
    s = cosine_similarity(embeddings[target], vec)
    if s > best_sim:
        best_sim = s
        best_word = word

print(f"Most similar to {target}")
print(best_word)
print(f"Similarity = {best_sim:.2f}")

Most similar to cat
lion
Similarity = 1.00


## Exercise 10 (Assignment): Mini NLP Pipeline

**Objective:** Combine multiple concepts into one application.

**Input**
```
I love NLP
I love AI
AI loves Python
Python loves data
```

**Tasks**
1. Build the vocabulary.
2. Count word frequencies.
3. Compute TF.
4. Build the co-occurrence matrix.
5. Detect sparse entries.
6. Apply Laplace smoothing to unigram probabilities.
7. Predict the next word after a user-entered word.
8. Display the top three most frequent words.

> Since this corpus is longer than the worksheet's illustrative example, the
> sparse-pair list is longer too (15 pairs instead of 2) — a sample plus the
> total count is printed for readability.


In [11]:
pipeline_corpus = [
    "I love NLP",
    "I love AI",
    "AI loves Python",
    "Python loves data",
]

# 1. Vocabulary
vocab10 = set()
for sentence in pipeline_corpus:
    vocab10.update(sentence.split())
print(f"Vocabulary Size : {len(vocab10)}")

# 2. Word frequencies
all_words = [w for sentence in pipeline_corpus for w in sentence.split()]
freq10 = Counter(all_words)

# 3. TF (overall, across whole corpus)
total_words10 = len(all_words)
tf10 = {w: c / total_words10 for w, c in freq10.items()}

# 4. Co-occurrence matrix (window size 1, per sentence)
vocab10_sorted = sorted(vocab10)
matrix10 = {w1: {w2: 0 for w2 in vocab10_sorted} for w1 in vocab10_sorted}
for sentence in pipeline_corpus:
    ws = sentence.split()
    for i, word in enumerate(ws):
        if i > 0:
            matrix10[word][ws[i - 1]] += 1
        if i < len(ws) - 1:
            matrix10[word][ws[i + 1]] += 1

# 5. Sparse entries
sparse_pairs10 = []
for i, w1 in enumerate(vocab10_sorted):
    for w2 in vocab10_sorted[i + 1:]:
        if matrix10[w1][w2] == 0:
            sparse_pairs10.append((w1, w2))

# 6. Laplace smoothing on unigram probabilities
vsize10 = len(vocab10)
laplace10 = {w: (c + 1) / (total_words10 + vsize10) for w, c in freq10.items()}

# 7. Predict next word (bigram model)
bigram_counts10 = defaultdict(Counter)
for sentence in pipeline_corpus:
    ws = sentence.split()
    for i in range(len(ws) - 1):
        bigram_counts10[ws[i]][ws[i + 1]] += 1

query_word = "love"
predictions10 = [w for w, _ in bigram_counts10[query_word].most_common(3)]

# 8. Top three most frequent words
top_words10 = freq10.most_common(3)

print("\nTop Words")
for w, c in top_words10:
    print(f"{w} : {c}")

print(f"\nPrediction after \"{query_word}\"")
for w in predictions10:
    print(w)

print("\nSparse Pairs")
for pair in sparse_pairs10[:5]:
    print(pair)
print(f"... ({len(sparse_pairs10)} sparse pairs total)")

Vocabulary Size : 7

Top Words
I : 2
love : 2
AI : 2

Prediction after "love"
NLP
AI

Sparse Pairs
('AI', 'I')
('AI', 'NLP')
('AI', 'Python')
('AI', 'data')
('I', 'NLP')
... (15 sparse pairs total)


## Exercise 11: Predict the Next Word Using Conditional Probability

**Objective:** Use conditional probability to predict the most likely next word
after a given word, using a small corpus.

**Background:** The conditional probability of a word $w_2$ given a previous
word $w_1$ is:

$$P(w_2 \mid w_1) = \frac{\text{Count}(w_1, w_2)}{\text{Count}(w_1)}$$

**Training Corpus**
```
I like NLP
I like Python
I like coffee
I love NLP
You like coffee
```

**Programming Task**
1. Read the training corpus.
2. Build bigram counts.
3. Calculate conditional probabilities for every bigram.
4. Ask the user to enter a word.
5. Display all possible next words and their conditional probabilities.
6. Predict the word with the highest conditional probability.

**Bonus Challenge:** display the top three predictions instead of only the best one.

**Concepts Learned:** Conditional Probability · Bigram Models


In [12]:
corpus11 = [
    "I like NLP",
    "I like Python",
    "I like coffee",
    "I love NLP",
    "You like coffee",
]

# Step 1: Bigram counts
bigram_counts11 = defaultdict(Counter)
for sentence in corpus11:
    ws = sentence.split()
    for i in range(len(ws) - 1):
        bigram_counts11[ws[i]][ws[i + 1]] += 1

# Step 2: Conditional probabilities  P(w2 | w1) = count(w1, w2) / count(w1)
cond_probs = {}
for w1, next_words in bigram_counts11.items():
    total = sum(next_words.values())
    cond_probs[w1] = {w2: c / total for w2, c in next_words.items()}

# In a script you'd use: query = input("Enter a word: ")
query = "like"
print(f"Enter a word: {query}")

if query in cond_probs:
    possibilities = sorted(cond_probs[query].items(), key=lambda x: x[1], reverse=True)

    print("\nPossible next words:")
    for word, prob in possibilities:
        print(f"{word} : {prob:.2f}")

    predicted = possibilities[0][0]
    print(f"\nPredicted next word: {predicted}")

    # Bonus: top three predictions
    print("\nTop Predictions")
    for i, (word, prob) in enumerate(possibilities[:3], start=1):
        print(f"{i}. {word} ({prob:.2f})")
else:
    print(f"No bigram data available for '{query}'.")

Enter a word: like

Possible next words:
coffee : 0.50
NLP : 0.25
Python : 0.25

Predicted next word: coffee

Top Predictions
1. coffee (0.50)
2. NLP (0.25)
3. Python (0.25)
